# Clase 9 — Tres niveles del software de ML

## Introducción 📖

En la clase anterior estudiamos la motivación y los principios de MLOps. Ahora veremos un proyecto de machine learning desde una perspectiva de ingeniería de software.

El desarrollo no ocurre en el vacío: comienza con **datos** que deben conocerse y prepararse; continúa con **modelos** que se entrenan, evalúan y empaquetan; y se integra mediante **código** que permite usar y observar las predicciones.

El objetivo de esta sesión es reconocer las piezas de un sistema de ML, comprender cómo se conectan y comparar formas comunes de llevar un modelo a una aplicación.

## Antes de comenzar

Actualiza tu copia del repositorio del curso:

```bash
git status
git pull
```

## Continuación de la Clase 8

En esta clase socializaremos las respuestas finales de la Clase 8. Revisa las tuyas y asegúrate de poder explicar la evidencia que utilizaste y la brecha que identificaste.

Durante la conversación con el grupo, ubicaremos las brechas en uno o más niveles del sistema: datos, modelo o código.

# 1. 🎯 Tres niveles del software de ML

La meta de un proyecto de machine learning no es solamente entrenar un algoritmo. Es construir un sistema capaz de producir predicciones útiles de manera reproducible y confiable. Para lograrlo, debe gestionar tres activos principales:

![Mapa de los tres niveles y sus divisiones](../assets/modulo-02-ciclo-mlops/clase-09/mapa-tres-niveles.svg)

*El diagrama resume las etapas de datos, modelo y código, y muestra que una predicción operable depende de las interfaces entre los tres niveles.*

- 📊 **Datos:** observaciones, variables, etiquetas y reglas de calidad.
- 🧠 **Modelo:** parámetros aprendidos, métricas, versiones y artefactos.
- 💻 **Código:** preparación, entrenamiento, inferencia, integración y operación.

Estos activos corresponden a tres áreas de trabajo: **Data Engineering**, **ML Model Engineering** y **Software Release Engineering**. MLOps conecta sus prácticas para diseñar, construir, desplegar y mantener sistemas de ML.

> Los tres niveles no son una escala de madurez. Son partes del mismo sistema y deben evolucionar de manera coordinada.

## 1.1 📊 Datos: pipelines de Data Engineering

Los datos son el punto de partida de cualquier flujo de ML. La calidad del modelo depende de qué observaciones se recolectan, cómo se interpretan y qué tan bien representan el problema real. La frase *garbage in, garbage out* resume una limitación importante: un algoritmo no corrige por sí solo datos incorrectos o poco representativos.

La ingeniería de datos suele consumir una parte considerable del trabajo porque exige construir, limpiar y transformar conjuntos reproducibles. Su pipeline reúne cuatro etapas:

1. ingesta de datos;
2. exploración y validación;
3. limpieza y transformación (*data wrangling*);
4. división en conjuntos para entrenamiento, validación y prueba.

El resultado esperado no es una colección de archivos sueltos, sino datos preparados cuyo origen y transformaciones puedan explicarse.

### 1.1.1 🚀 Data Ingestion

La ingesta reúne datos desde las fuentes necesarias y los coloca en un espacio donde puedan analizarse sin alterar los originales. Las fuentes pueden ser bases internas o externas, sistemas transaccionales, almacenes analíticos, archivos, APIs, sensores o plataformas de procesamiento distribuido. También puede incluir datos sintéticos o enriquecimiento con fuentes adicionales.

Una ingesta responsable documenta:

- **fuente y procedencia:** quién produce los datos, cuándo y bajo qué condiciones;
- **volumen y almacenamiento:** cuánto espacio requieren y dónde se conservarán;
- **formato:** cómo obtener una copia manipulable sin modificar el original;
- **respaldo:** cómo recuperar la fuente original;
- **privacidad y acceso:** qué información es sensible y quién puede utilizarla;
- **metadatos:** tamaño, periodo, formato, esquema y permisos;
- **reserva de prueba:** qué información permanecerá fuera del desarrollo del modelo.

> **Ejemplo del curso — Green Taxi:** la fuente es NYC TLC. Conservamos el periodo de cada viaje y usamos marzo de 2026 para entrenamiento y abril de 2026 para validación. Los archivos originales no se sustituyen por los datos preparados.

### 1.1.2 🔍 Exploración y validación

La exploración permite conocer el contenido y la estructura antes de modelar. El perfilado registra, por ejemplo, nombres, tipos, valores faltantes, valores únicos, mínimos, máximos, promedios y distribuciones. Las visualizaciones ayudan a reconocer patrones que una tabla de resumen puede ocultar.

La validación convierte ese conocimiento en condiciones comprobables. Una regla puede preguntar si una columna existe, si un identificador pertenece a un catálogo, si faltan valores importantes o si un rango es físicamente posible. Estas reglas deben responder al significado del dato, no sólo a su tipo en Python.

Un notebook puede documentar la exploración inicial, pero las comprobaciones que se repetirán deben transformarse en código reutilizable. También es necesario identificar el **target** en tareas supervisadas y revisar relaciones entre atributos sin asumir que correlación implica causalidad.

> **Ejemplo del curso — Green Taxi:** verificamos que la duración sea positiva, que la distancia se encuentre entre 0 y 100 km, que haya entre 1 y 6 pasajeros, que la hora esté entre 0 y 23 y que las zonas tengan identificadores válidos entre 1 y 265.

### 1.1.3 🛠️ Limpieza de datos (*Data Wrangling*)

El wrangling transforma los datos explorados en una representación adecuada para el modelo. Las operaciones comunes incluyen:

- corregir o excluir valores atípicos cuando exista una justificación;
- tratar valores faltantes mediante imputación o exclusión;
- retirar campos irrelevantes o que provoquen fuga de información;
- cambiar tipos y formatos;
- combinar o descomponer campos;
- crear variables derivadas;
- escalar o codificar variables cuando el algoritmo lo requiera.

Las transformaciones deben vivir en scripts o funciones reutilizables. Si el entrenamiento y la inferencia preparan la misma variable de maneras diferentes, el sistema puede aceptar una solicitud y aun así producir una predicción incorrecta.

> **Ejemplo del curso — Green Taxi:** a partir de columnas crudas construimos `distancia_km`, `pasajeros`, `hora_recoleccion`, `zona_origen`, `zona_destino` y el target `duracion_minutos`. Las zonas se codifican dentro del pipeline del modelo para reutilizar exactamente la misma transformación.

In [ ]:
FEATURES = [
    "distancia_km",
    "pasajeros",
    "hora_recoleccion",
    "zona_origen",
    "zona_destino",
]
TARGET = "duracion_minutos"

solicitud = {
    "distancia_km": 4.2,
    "pasajeros": 2,
    "hora_recoleccion": 18,
    "zona_origen": 75,
    "zona_destino": 42,
}

faltantes = [feature for feature in FEATURES if feature not in solicitud]
print("Contrato completo:", not faltantes)


### 1.1.4 🔀 División de datos

Los datos se dividen para evitar evaluar un modelo con las mismas observaciones que utilizó para aprender:

- **entrenamiento:** ajusta los parámetros del modelo;
- **validación:** compara alternativas y apoya decisiones de selección;
- **prueba:** se reserva para estimar el desempeño final sobre observaciones no utilizadas durante el desarrollo.

Una proporción como 80/10/10 puede ser un punto de partida, pero no es una regla universal. La división debe respetar la estructura del problema: tiempo, grupos, personas, ubicaciones o cualquier relación que pueda provocar fuga de información.

> **Ejemplo del curso — Green Taxi:** usamos marzo de 2026 para entrenamiento y abril de 2026 para validación. Una división temporal representa mejor el uso futuro que mezclar aleatoriamente viajes de ambos meses.

## 1.2 🛠️ Modelo: pipelines de Machine Learning

El pipeline de model engineering transforma los datos preparados en un modelo que puede evaluarse y consumirse. Incluye cuatro etapas distintas:

1. **Model Training**;
2. **Model Evaluation**;
3. **Model Testing**;
4. **Model Packaging**.

Separarlas ayuda a evitar una confusión común: obtener un modelo entrenado no significa que esté listo para utilizarse.

### 1.2.1 📚 Entrenamiento del modelo

Entrenar significa aplicar un algoritmo a los datos de entrenamiento para estimar sus parámetros. Antes y durante este proceso pueden existir dos tipos de ingeniería:

**Ingeniería de características**

- discretizar variables continuas cuando tenga sentido;
- descomponer fechas, categorías u otros campos;
- aplicar transformaciones matemáticas;
- combinar variables para crear representaciones útiles;
- estandarizar o normalizar cuando el algoritmo sea sensible a la escala.

**Ingeniería del modelo**

- versionar y revisar el código que define cada alternativa;
- comparar familias de modelos apropiadas para la tarea;
- medir el desempeño con un procedimiento consistente;
- analizar qué tipos de errores comete cada modelo;
- revisar o ampliar las features cuando la evidencia lo justifique;
- ajustar hiperparámetros sin utilizar el conjunto de prueba;
- considerar ensambles cuando la mejora compense su complejidad.

> **Ejemplo del curso — Green Taxi:** el pipeline codifica `zona_origen` y `zona_destino` con `OneHotEncoder` y ajusta una regresión lineal. Esta alternativa debe compararse contra el baseline antes de considerarse una mejora.

### 1.2.2 ✔️ Model Evaluation

La evaluación comprueba si el modelo satisface el objetivo técnico y apoya la necesidad del proyecto. Exige una métrica definida, un conjunto de validación y un criterio para decidir si una alternativa mejora lo existente. Una sola cifra no basta: también conviene analizar en qué casos aparecen los errores y si el desempeño es consistente para grupos relevantes.

> **Ejemplo del curso — Green Taxi:** usamos RMSE expresado en minutos. El candidato se compara con el baseline sobre viajes de abril y no se promueve sólo porque el entrenamiento terminó sin errores.

### 1.2.3 🧪 Model Testing

Después de seleccionar una alternativa, el conjunto de prueba reservado permite estimar su error de generalización. Esta **prueba de aceptación del modelo** se ejecuta bajo condiciones definidas antes de observar el resultado. También pueden probarse el esquema de entrada, la compatibilidad del artefacto y casos límite.

> **Ejemplo del curso — Green Taxi:** abril funciona actualmente como validación. Un sistema posterior necesitaría declarar un periodo de prueba que no haya intervenido en la selección del modelo.

### 1.2.4 📦 Model Packaging

Empaquetar consiste en exportar el modelo entrenado en una forma que otra aplicación pueda cargar. Existen formatos y estrategias como ONNX, PMML, PFA, archivos binarios o serialización específica de una biblioteca. La elección depende de compatibilidad, seguridad y entorno de ejecución. Un archivo `pickle`, por ejemplo, sólo debe cargarse desde una fuente confiable.

El artefacto útil necesita contexto: versión, target, orden de features, transformaciones, biblioteca compatible, métrica y datos con los que se evaluó.

> **Ejemplo del curso — Green Taxi:** el artefacto contiene el pipeline entrenado, las cinco features en orden, la versión `green-taxi-2026-03-linear-zonas-1` y su RMSE de validación.

In [ ]:
artefacto = {
    "version": "green-taxi-2026-03-linear-zonas-1",
    "target": TARGET,
    "features": FEATURES,
    "evaluado_con": "viajes de abril de 2026",
    "metrica": "RMSE",
}

print(artefacto)


### 1.2.5 🔄 Formas de operar un modelo

Los flujos de ML pueden compararse mediante dos decisiones independientes.

**¿Cómo se entrena?**

- **Offline, batch o estático:** aprende con un conjunto ya recolectado y permanece sin cambios hasta el siguiente entrenamiento. Su desempeño puede deteriorarse si cambian los datos; esto suele llamarse *model decay* o degradación del modelo.
- **Online, dinámico o incremental:** actualiza sus parámetros regularmente conforme llegan nuevos datos o pequeños lotes. Requiere controles adicionales porque los datos erróneos pueden afectar rápidamente las versiones posteriores.

**¿Cómo produce predicciones?**

- **Batch:** calcula muchas predicciones en una ejecución programada y puede almacenar los resultados.
- **Bajo demanda:** responde con los datos disponibles en el momento de cada solicitud.

Antes de revisar la figura, identifica sus ejes y localiza un sistema que entrene offline y prediga bajo demanda.

![Matriz que cruza entrenamiento estático o dinámico con predicción batch o bajo demanda](../assets/modulo-02-ciclo-mlops/clase-09/model-serving-pattern.png)

*La figura organiza cuatro escenarios frecuentes. Sus etiquetas son ejemplos, no una taxonomía rígida; AutoML automatiza partes del pipeline de modelado y no define por sí solo cómo se sirve una predicción.*

#### 1.2.5.1 🔮 Forecast o batch prediction

Un modelo se entrena con datos recolectados y después calcula predicciones para un lote. Es apropiado cuando la respuesta inmediata no es necesaria. Ejemplos: proyectar ventas mensuales o evaluar semanalmente una cartera de solicitudes de crédito.

En este patrón deben definirse la frecuencia del lote, dónde se almacenan los resultados y cuánto tiempo siguen siendo válidos.

#### 1.2.5.2 🌐 Web Service/API

El modelo se entrena offline, pero responde una solicitud a la vez con datos disponibles en ese momento. El servicio permanece estable hasta que una nueva versión del modelo se evalúa y despliega. Ejemplos: detección de fraude durante una transacción o estimación de duración al solicitar un viaje.

![Entrenamiento que produce un artefacto y predicción mediante cliente y API](../assets/modulo-02-ciclo-mlops/clase-09/model-serving-as-microservice.png)

*Observa la separación: entrenamiento produce un artefacto; el servicio lo carga y expone un contrato para el cliente.*

> **Ejemplo del curso — Green Taxi:** `POST /predicciones` recibe las cinco features, valida sus rangos, carga el artefacto y devuelve una duración estimada junto con la versión del modelo.

#### 1.2.5.3 🚀 Online Learning

El aprendizaje online —también llamado incremental— incorpora continuamente observaciones o pequeños lotes. Se utiliza cuando el comportamiento cambia con rapidez y el sistema necesita adaptarse. Recomendaciones, precios o señales de sensores son ejemplos posibles.

Su complejidad no está sólo en entrenar con frecuencia: es necesario validar el flujo, evaluar versiones, controlar retrocesos y evitar que datos erróneos degraden el sistema.

![Ciclo de aprendizaje online con entrenamiento evaluación y despliegue](../assets/modulo-02-ciclo-mlops/clase-09/online-learning.png)

*Aunque el ciclo sea frecuente, evaluación y decisión de despliegue siguen siendo etapas separadas.*

> **Ejemplo de contraste:** Green Taxi no aprende durante cada solicitud. Su modelo permanece fijo hasta ejecutar nuevamente el pipeline de entrenamiento.

#### 1.2.5.4 ⚙️ AutoML

AutoML automatiza tareas como selección de algoritmos, transformaciones o ajuste de hiperparámetros. Herramientas comerciales y de código abierto pueden producir múltiples candidatos a partir de un conjunto de datos.

La automatización no define el problema, no garantiza datos confiables y no decide por sí sola si una métrica satisface la necesidad real. Los modelos generados todavía deben evaluarse, documentarse, empaquetarse y operarse. Plataformas como Google Cloud, Azure Machine Learning, H2O, DataRobot y auto-sklearn ofrecen distintas variantes.

## 1.3 🚀 Código: deployment pipelines

La entrega de un sistema de ML incluye tres responsabilidades:

1. **Model Serving:** poner una predicción a disposición de una aplicación o proceso.
2. **Model Performance Monitoring:** observar si el sistema y el desempeño cambian.
3. **Model Performance Logging:** registrar evidencia para investigar lo ocurrido.

La inferencia necesita un modelo, un entorno capaz de ejecutarlo y datos de entrada compatibles. Además del servicio de predicción, un sistema completo puede necesitar código para reentrenar, validar y promover nuevas versiones.

> **Ejemplo del curso — Green Taxi:** FastAPI sirve la predicción; la versión, latencia, errores de validación y códigos de respuesta serían señales útiles. Medir desempeño requiere obtener después la duración real del viaje.

### 1.3.1 📊 Model Serving Patterns

Existen tres patrones frecuentes para integrar un modelo en software. La elección depende del consumidor, la frecuencia de actualización, la latencia y el acoplamiento permitido.

#### 1. Model-as-Service

El modelo y su intérprete viven en un servicio independiente. Otras aplicaciones solicitan predicciones mediante un contrato como REST o gRPC. Varios clientes pueden compartir la misma versión, aunque pasan a depender de la red y de la disponibilidad del servicio.

![Aplicación que consulta un servicio independiente con el modelo](../assets/modulo-02-ciclo-mlops/clase-09/model-as-a-service.png)

*Identifica el límite entre la aplicación cliente y el servicio del modelo.*

> **Ejemplo del curso:** la API de Green Taxi sigue este patrón.

#### 2. Model-as-Dependency

El modelo empaquetado se incluye como dependencia de la aplicación, que invoca directamente su método de predicción. Reduce una llamada de red, pero acopla la versión del modelo, las bibliotecas y el despliegue de la aplicación. Puede ser útil para procesos sencillos o controlados.

![Aplicación que contiene el modelo como dependencia](../assets/modulo-02-ciclo-mlops/clase-09/model-as-a-dependency.png)

*En este patrón, actualizar el modelo normalmente implica volver a desplegar la aplicación que lo contiene.*

#### 3. Precompute Serving Pattern

El modelo calcula por adelantado las predicciones de un lote y las guarda en una base de datos. Cuando llega una consulta, la aplicación recupera el resultado almacenado en lugar de ejecutar el modelo. Es útil cuando las entradas son conocidas, pero exige definir cuándo caducan o se recalculan las predicciones.

![Predicciones de un lote almacenadas para consultas posteriores](../assets/modulo-02-ciclo-mlops/clase-09/precompute.png)

*Observa que la consulta y la inferencia ocurren en momentos diferentes.*

### 1.3.2 🌐 Estrategias de despliegue

Un patrón de serving explica cómo se conecta el modelo con el consumidor. Una estrategia de despliegue explica cómo se empacan y ejecutan el código, las dependencias y el artefacto.

#### Contenedores Docker

Un contenedor puede reunir el código de inferencia, el modelo y sus dependencias en un entorno reproducible. Puede ejecutarse localmente, en infraestructura propia o en la nube, y plataformas como Kubernetes administran conjuntos de contenedores cuando la escala lo requiere.

La contenedorización es común, pero la inferencia no siempre es ligera, sin estado o idempotente: esas propiedades deben diseñarse y comprobarse para cada sistema.

![Modelo y servicio empacados en Docker](../assets/modulo-02-ciclo-mlops/clase-09/deploy-docker.png)

*Observa qué elementos deben permanecer compatibles dentro del contenedor.*

#### Funciones serverless

Una plataforma serverless ejecuta un punto de entrada administrado por el proveedor. La aplicación y sus dependencias se empaquetan para ese entorno, mientras la plataforma gestiona parte del aprovisionamiento y escalamiento. AWS Lambda, Azure Functions y Google Cloud Functions son ejemplos.

Deben evaluarse límites de tamaño, memoria, tiempo de inicio, duración, costo y compatibilidad del artefacto. Serverless reduce tareas de infraestructura, pero no elimina la validación, el versionado ni el monitoreo.

![Modelo desplegado como función administrada](../assets/modulo-02-ciclo-mlops/clase-09/deploy-serverless.png)

*Identifica el punto de entrada y las responsabilidades que conserva el equipo.*

## Actividad calificable en clase — Mapa inicial del sistema de ML

Con tu equipo, construyan `docs/mapa-sistema-ml.md` en el repositorio privado del proyecto. Este documento será una guía de diseño que podrá cambiar durante el proyecto; no representa una arquitectura definitiva.

Partan de la evidencia que ya tienen en el EDA. En cada apartado distingan qué conocen, qué decisión inicial proponen, qué suponen y qué pregunta necesitan responder después:

- contexto y consumidor;
- Data Pipeline: fuente, unidad de observación, target, features candidatas, validaciones, transformaciones y división;
- Model Pipeline: tarea, baseline, métrica inicial, aceptación, prueba y artefacto esperado;
- Code Deployment Pipeline: componentes probables para entrenamiento e inferencia, consumidor, interfaces y señales;
- forma de entrenamiento e inferencia;
- comparación de los tres patrones de serving y elección inicial;
- diagrama del recorrido completo;
- riesgo prioritario y siguiente incremento.

No se espera que todas las respuestas estén cerradas. Un supuesto claro o una pregunta bien formulada es mejor que inventar una decisión sin evidencia. Trabajen en la rama `docs/mapa-sistema-ml`, abran un PR hacia `main`, revísenlo, fusiónenlo y ciérrenlo. Cada integrante entrega en Canvas la URL del mismo PR fusionado. Consulta las instrucciones y la rúbrica en [Actividad en clase 3](../docs/tareas/actividad-clase-09-mapa-sistema-ml.md).

# Conclusión 📝

Un proyecto de machine learning se convierte en un sistema de software cuando coordina tres niveles:

- en **datos**, ingesta, exploración, validación, transformación y división;
- en **modelo**, entrenamiento, evaluación, prueba y packaging;
- en **código**, serving, logging, monitoring y despliegue.

La dificultad de operar ML no reside únicamente en el algoritmo, sino en las interfaces y decisiones que lo rodean. Antes de cerrar, explica con tus propias palabras dónde termina cada nivel y qué contrato lo conecta con el siguiente.

## Referencias y atribución de figuras

- INNOQ, [Three Levels of ML Software](https://ml-ops.org/content/three-levels-of-ml-software), CC BY 4.0. Figuras y atribuciones en [`assets/modulo-02-ciclo-mlops/clase-09/README.md`](../assets/modulo-02-ciclo-mlops/clase-09/README.md).
- Google Cloud, [MLOps: Continuous delivery and automation pipelines in machine learning](https://docs.cloud.google.com/architecture/mlops-continuous-delivery-and-automation-pipelines-in-machine-learning).
- Sculley et al., [Hidden Technical Debt in Machine Learning Systems](https://research.google/pubs/hidden-technical-debt-in-machine-learning-systems/).